In [1]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from searcharray import SearchArray
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [2]:
MAX_CHARACTERS = 30_000
NAMESPACE = uuid.NAMESPACE_URL

def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def organize_repo_data(repo_data: List[Dict[Any, Any]]) -> Tuple[List[str], List[Dict[str, Union[str, int]]], List[str]]:
    ids = []
    metadata = []
    docs = []
    for repo in repo_data:
        id = repo_to_uuid(repo['repo'])
        description = repo['description']
        topics = repo['topics']
        doc_source = repo['doc_source']
        stars = repo['stars']
        url = repo['url']
        language = repo['language']
        # append to lists for DB consumption
        ids.append(id)
        metadata.append({
            'id': id,
            'repo': repo['repo'],
            'description': description,
            'topics': topics,
            'language': language,
            'doc_source': doc_source,
            'stars': stars,
            'url': url,
        })
        topics_str = f"Topics: {','.join(topics)}\n"
        docs.append(topics_str + normalize_docs(repo['docs']))
    return ids, metadata, docs


def preprocess_text(text: str | None) -> List[str]:
    """borrowed from Ask Brave search for common preprocessing/tokenizer steps before TF-IDF"""
    if text is None:
        return []
    # Lowercase
    text = text.lower()
    # Remove punctuation and digits
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

In [3]:
# load cached github data
with open("../data/cached/github_data.pkl", "rb") as f:
    starred_repo_data = pickle.load(f)

In [4]:
ids, metadata, docs = organize_repo_data(starred_repo_data)

In [5]:
metadata[0]

{'id': 'a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb',
 'repo': 'pymc-devs/pytensor',
 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.',
 'topics': ['ai',
  'bayesian-inference',
  'computational-science',
  'deep-learning',
  'statistics'],
 'language': 'Python',
 'doc_source': 'readme',
 'stars': 594,
 'url': 'https://github.com/pymc-devs/pytensor'}

In [6]:
docs[0]

'Topics: ai,bayesian-inference,computational-science,deep-learning,statistics\n.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensorRGB.svg\n :height: 100px\n :alt: PyTensor logo\n :align: center\n|Tests Status| |Coverage|\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for PyMC .\nFeatures\n========\n- A hackable, pure-Python codebase\n- Extensible graph framework suitable for rapid development of custom operators and symbolic optimizations\n- Implements an extensible graph transpilation framework that currently provides\n compilation via C, JAX , and Numba \n- Contrary to PyTorch and TensorFlow, PyTensor maintains a static graph which can be modified in-place to\n allow for advanced optimizations\nGetting started\n===============\n.. code-block:: python\n import pytensor\n from pytensor import tensor as pt\n # D

In [7]:
starred_repo_data[0]

{'repo': 'pymc-devs/pytensor',
 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.',
 'topics': ['ai',
  'bayesian-inference',
  'computational-science',
  'deep-learning',
  'statistics'],
 'stars': 594,
 'language': 'Python',
 'url': 'https://github.com/pymc-devs/pytensor',
 'doc_source': 'readme',
 'docs': [{'name': 'README.rst',
   'path': 'README.rst',
   'size': 4199,
   'content': '.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensor_RGB.svg\n    :height: 100px\n    :alt: PyTensor logo\n    :align: center\n\n|Tests Status| |Coverage|\n\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for `PyMC <https://github.com/pymc-devs/pymc>`__.\n\nFeatures\n========\n\n- A hackable, pure-Python codebase\n- Extensible graph frame

In [8]:
df = pd.DataFrame(starred_repo_data)
df.head()

,repo,description,topics,stars,language,url,doc_source,docs
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","[ai, bayesian-inference, computational-science...",594,Python,https://github.com/pymc-devs/pytensor,readme,"[{'name': 'README.rst', 'path': 'README.rst', ..."
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,[],315,Jupyter Notebook,https://github.com/Pyomo/pyomo-gallery,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,[],41,Python,https://github.com/Pyomo/pyomo-tutorials,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,[],120,Jupyter Notebook,https://github.com/norhum/reinforcement-learni...,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"[data-mining, natural-language-processing, pyt...",4464,HTML,https://github.com/deanmalmgren/textract,readme,"[{'name': 'README.rst', 'path': 'README.rst', ..."


In [9]:
df['id'] = df['repo'].apply(repo_to_uuid)
df.head()

,repo,description,topics,stars,language,url,doc_source,docs,id
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","[ai, bayesian-inference, computational-science...",594,Python,https://github.com/pymc-devs/pytensor,readme,"[{'name': 'README.rst', 'path': 'README.rst', ...",a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,[],315,Jupyter Notebook,https://github.com/Pyomo/pyomo-gallery,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",93cef65a-529e-5666-b09f-751e76cc4621
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,[],41,Python,https://github.com/Pyomo/pyomo-tutorials,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",3a4e6d0c-cbbe-5ab5-b091-70f35b883ff8
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,[],120,Jupyter Notebook,https://github.com/norhum/reinforcement-learni...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",aef234b1-aa71-5aef-ab2a-b63d9a25b3b4
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"[data-mining, natural-language-processing, pyt...",4464,HTML,https://github.com/deanmalmgren/textract,readme,"[{'name': 'README.rst', 'path': 'README.rst', ...",0b546aaa-07f3-567a-8cb7-04e60e9182ca


In [10]:
def process_topics_column(
    df: pd.DataFrame,
    col: str,
    *,
    sep: str = " ",
    slug_sep: str = "-",
    fillna: str = "",
) -> pd.Series:
    """
    Process a DataFrame column whose cells are either a list of strings or a
    plain string, and return a clean, flat string per row.

    Each string in a cell is:
      1. Stripped of leading/trailing whitespace.
      2. Split on `slug_sep` (default "-") so hyphenated slugs like
         "natural-language-processing" become "natural language processing".
      3. Lower-cased.

    The resulting tokens from all strings in a cell are then joined with `sep`
    (default " "), giving a single string per row.  Empty / null cells are
    replaced with `fillna` (default "").

    Args:
        df:       Input DataFrame.
        col:      Name of the column to process.
        sep:      Token join separator for the final string.
        slug_sep: Separator used to split individual slug strings (e.g. "-").
        fillna:   Value to use for rows that are null or produce an empty result.

    Returns:
        A pd.Series of processed strings aligned to `df`'s index.

    Example:
        >>> process_string_column(df, "topics").head()
        0    ai bayesian inference computational science deep learning statistics
        1
        2
        dtype: object
    """

    def _process_cell(cell) -> str:
        if cell is None:
            return fillna

        # Accept both list[str] and a bare str
        items: list[str] = cell if isinstance(cell, list) else [cell]

        tokens: list[str] = []
        for item in items:
            # Split on the slug separator and lower-case each sub-token
            tokens.extend(part.lower() for part in str(item).strip().split(slug_sep) if part)

        return sep.join(tokens) if tokens else fillna

    return df[col].apply(_process_cell)


# --- Demo on the `topics` column ---
df["topics_processed"] = process_topics_column(df, "topics")
df[["repo", "topics", "topics_processed"]].head(10)

,repo,topics,topics_processed
0,pymc-devs/pytensor,"[ai, bayesian-inference, computational-science...",ai bayesian inference computational science de...
1,Pyomo/pyomo-gallery,[],
2,Pyomo/pyomo-tutorials,[],
3,norhum/reinforcement-learning-from-scratch,[],
4,deanmalmgren/textract,"[data-mining, natural-language-processing, pyt...",data mining natural language processing python...
5,ashishps1/awesome-engineering-articles,"[ai, data-engineering, database, distributed-s...",ai data engineering database distributed syste...
6,microsoft/markitdown,"[autogen, autogen-extension, langchain, markdo...",autogen autogen extension langchain markdown m...
7,pyinfra-dev/pyinfra,"[cloud-management, configuration-management, h...",cloud management configuration management high...
8,cognica-io/bayesian-bm25,"[bayesian-bm25, bayesian-inference, bm25, hybr...",bayesian bm25 bayesian inference bm25 hybrid s...
9,chartbeat-labs/textacy,"[natural-language-processing, nlp, python, spacy]",natural language processing nlp python spacy


In [11]:
df['topics_tokenized'] = SearchArray.index(df['topics_processed'], tokenizer=preprocess_text)

2026-03-12 16:40:48,498 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-12 16:40:48,499 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-12 16:40:48,499 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-12 16:40:50,149 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-12 16:40:50,150 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-12 16:40:50,151 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-12 16:40:50,153 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-12 16:40:50,155 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-12 16:40:50,155 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-12 16:40:50,157 - searcharray.indexing - INFO - Indexing from tokenization complete


In [12]:
df['description_tokenized'] = SearchArray.index(df['description'], tokenizer=preprocess_text)
df['repo_tokenized'] = SearchArray.index(df['repo'], tokenizer=preprocess_text)

2026-03-12 16:40:50,166 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-12 16:40:50,167 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-12 16:40:50,167 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-12 16:40:50,392 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-12 16:40:50,392 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-12 16:40:50,393 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-12 16:40:50,395 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-12 16:40:50,397 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-12 16:40:50,398 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-12 16:40:50,399 - searcharray.indexing - INFO - Indexing from tokenization complete
2026-03-12 16:40:50,401 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-12 16:40:50,402 - searcharray.indexing - INFO - 0 Batch Sta

In [13]:
df[['repo','description','topics', 'repo_tokenized']].head()

,repo,description,topics,repo_tokenized
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","[ai, bayesian-inference, computational-science...","Terms({'devs', 'pytensor', 'pymc'})"
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,[],"Terms({'gallery', 'pyomo'})"
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,[],"Terms({'tutorial', 'pyomo'})"
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,[],"Terms({'scratch', 'learning', 'norhum', 'reinf..."
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"[data-mining, natural-language-processing, pyt...","Terms({'deanmalmgren', 'textract'})"


In [14]:
df['doc_source'].value_counts()

doc_source
readme    2041
Name: count, dtype: int64

In [15]:
df['docs'][0]

[{'name': 'README.rst',
  'path': 'README.rst',
  'size': 4199,
  'content': '.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensor_RGB.svg\n    :height: 100px\n    :alt: PyTensor logo\n    :align: center\n\n|Tests Status| |Coverage|\n\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for `PyMC <https://github.com/pymc-devs/pymc>`__.\n\nFeatures\n========\n\n- A hackable, pure-Python codebase\n- Extensible graph framework suitable for rapid development of custom operators and symbolic optimizations\n- Implements an extensible graph transpilation framework that currently provides\n  compilation via C, `JAX <https://github.com/google/jax>`__, and `Numba <https://github.com/numba/numba>`__\n- Contrary to PyTorch and TensorFlow, PyTensor maintains a static graph which can be modified in-place to\n  allow for advanced op

In [16]:
df['docs'][0]

[{'name': 'README.rst',
  'path': 'README.rst',
  'size': 4199,
  'content': '.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensor_RGB.svg\n    :height: 100px\n    :alt: PyTensor logo\n    :align: center\n\n|Tests Status| |Coverage|\n\n|Project Name| is a Python library that allows one to define, optimize, and\nefficiently evaluate mathematical expressions involving multi-dimensional arrays.\nIt provides the computational backend for `PyMC <https://github.com/pymc-devs/pymc>`__.\n\nFeatures\n========\n\n- A hackable, pure-Python codebase\n- Extensible graph framework suitable for rapid development of custom operators and symbolic optimizations\n- Implements an extensible graph transpilation framework that currently provides\n  compilation via C, `JAX <https://github.com/google/jax>`__, and `Numba <https://github.com/numba/numba>`__\n- Contrary to PyTorch and TensorFlow, PyTensor maintains a static graph which can be modified in-place to\n  allow for advanced op

In [17]:
# Concatenate content from all docs in the list
df['content'] = df['docs'].apply(lambda x: '\n\n'.join(doc['content'] for doc in x) if x else None)

In [18]:
df[['repo','description','description_tokenized','content']].head()

,repo,description,description_tokenized,content
0,pymc-devs/pytensor,"PyTensor allows you to define, optimize, and e...","Terms({'involving', 'expression', 'efficiently...",.. image:: https://cdn.rawgit.com/pymc-devs/py...
1,Pyomo/pyomo-gallery,A collection of Pyomo examples,"Terms({'pyomo', 'collection', 'example'})",[![Project Status: Inactive – The project has ...
2,Pyomo/pyomo-tutorials,A modern set of Pyomo tutorials,"Terms({'modern', 'set', 'tutorial', 'pyomo'})",# Pyomo Tutorials\n\nThis repository hosts a m...
3,norhum/reinforcement-learning-from-scratch,RL for total beginners,"Terms({'beginner', 'total'})",# Reinforcement Learning From Scratch\n\nWelco...
4,deanmalmgren/textract,extract text from any document. no muss. no fuss.,"Terms({'extract', 'fuss', 'text', 'document', ...",.. NOTES FOR CREATING A RELEASE:\n..\n.. * b...


In [19]:
# normalize content
df["content_normalized"] = df["content"].map(
    lambda x: normalize_text_pipeline(x[:MAX_CHARACTERS]) if pd.notna(x) else x
)

In [20]:
df['content_normalized'][10]

"GitButler\n \n \n Git, but better.\n \n GitButler is a modern Git-based version control interface with both a GUI and CLI built from the ground up for AI-powered workflows.\n \n \n Website\n - \n Blog\n - \n Docs\n - \n Downloads\n \n \n \n Our beautiful GUI\n \n Our amazing but CLI\n \n[![TWEET][s1]][l1] [\n![BLUESKY][s8]][l8] [![DISCORD][s2]][l2]\n[![CI][s0]][l0] [![INSTA][s3]][l3] [![YOUTUBE][s5]][l5] [![DEEPWIKI][s7]][l7]\n[s0]: https://github.com/gitbutlerapp/gitbutler/actions/workflows/push.yaml/badge.svg\n[l0]: https://github.com/gitbutlerapp/gitbutler/actions/workflows/push.yaml\n[s1]: https://img.shields.io/badge/Twitter-black?logo=x&logoColor=white\n[l1]: https://twitter.com/intent/follow?screen_name=gitbutler\n[s2]: https://img.shields.io/discord/1060193121130000425?label=Discord&color=5865F2\n[l2]: https://discord.gg/MmFkmaJ42D\n[s3]: https://img.shields.io/badge/Instagram-E4405F?logo=instagram&logoColor=white\n[l3]: https://www.instagram.com/gitbutler/\n[s5]: https://img.

In [21]:
# tokenize content now
df['content_tokenized'] = SearchArray.index(df['content_normalized'], tokenizer=preprocess_text)

2026-03-12 16:40:53,218 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-12 16:40:53,218 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-12 16:40:53,219 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-12 16:41:00,334 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-12 16:41:00,337 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-12 16:41:00,344 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-12 16:41:00,547 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-12 16:41:00,603 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-12 16:41:00,604 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-12 16:41:00,623 - searcharray.indexing - INFO - Indexing from tokenization complete


In [22]:
df['score'] = df['content_tokenized'].array.score("awesome")

In [23]:
df.sort_values('score', ascending=False).head(10)

,repo,description,topics,stars,language,url,doc_source,docs,id,topics_processed,topics_tokenized,description_tokenized,repo_tokenized,content,content_normalized,content_tokenized,score
1533,MarcSkovMadsen/awesome-streamlit,The purpose of this project is to share knowle...,"[analytics, apps, awesome-list, data, data-sci...",2243,HTML,https://github.com/MarcSkovMadsen/awesome-stre...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",9369c17d-24eb-5696-a2a6-f992e65cb2bb,analytics apps awesome list data data science ...,"Terms({'apps', 'model', 'python', 'awesome', '...","Terms({'knowledge', 'share', 'awesome', 'proje...","Terms({'awesome', 'streamlit', 'marcskovmadsen'})",# Awesome Streamlit [![Awesome](https://cdn.ra...,Awesome Streamlit ![Awesome](https://github.co...,"Terms({'magic', 'youtu', 'madsen', 'black', 'l...",2.460909
1516,ramitsurana/awesome-kubernetes,A curated list for awesome kubernetes sources ...,"[aws, azure, books, cloud-providers, deploy-ku...",15825,Shell,https://github.com/ramitsurana/awesome-kubernetes,readme,"[{'name': 'README.md', 'path': 'docs/README.md...",f283f1aa-2254-5605-9630-80c59c07024b,aws azure books cloud providers deploy kuberne...,"Terms({'book', 'resource', 'product', 'learnin...","Terms({'curated', 'awesome', 'list', 'ship', '...","Terms({'awesome', 'kubernetes', 'ramitsurana'})",Awesome-Kubernetes\n==========================...,Awesome-Kubernetes\n==========================...,"Terms({'management', 'explained', 'vyas', 'lic...",2.429229
169,LincolnBurrows2017/gauss-awesome-recommender-s...,gauss-awesome-recommender-system-engine,[],122,Python,https://github.com/LincolnBurrows2017/gauss-aw...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",91abb07c-fa69-53c1-8e87-32235ac4a2a4,,Terms(set()),"Terms({'recommender', 'engine', 'system', 'awe...","Terms({'recommender', 'engine', 'system', 'awe...",# Gauss: Awesome Recommender System Engine 🚀\n...,Gauss: Awesome Recommender System Engine 🚀\n!G...,"Terms({'management', 'mostpopular', 'detail', ...",2.401310
1950,jnv/lists,The definitive list of lists (of lists) curate...,"[awesome, awesome-list, curated-lists, lists, ...",11002,None,https://github.com/jnv/lists,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",6a6a505a-45ec-5402-b338-aea20a0047fd,awesome awesome list curated lists lists resou...,"Terms({'awesome', 'resource', 'curated', 'list'})","Terms({'curated', 'github', 'elsewhere', 'defi...","Terms({'jnv', 'list'})","# Lists\n\nList of useful, silly and [awesome]...","Lists\nList of useful, silly and awesome lists...","Terms({'input', 'management', 'belarusian', 'n...",2.398078
61,sickn33/antigravity-awesome-skills,The Ultimate Collection of 900+ Agentic Skills...,"[agentic-skills, ai-agents, antigravity, auton...",18096,Python,https://github.com/sickn33/antigravity-awesome...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",93032f3d-079a-5cb4-bbe7-c25e58755229,agentic skills ai agents antigravity autonomou...,"Terms({'security', 'agent', 'autonomous', 'age...","Terms({'agent', 'agentic', 'antigravity', 'ant...","Terms({'awesome', 'skill', 'sickn', 'antigravi...",# 🌌 Antigravity Awesome Skills: 968+ Agentic S...,🌌 Antigravity Awesome Skills: 968+ Agentic Ski...,"Terms({'net', 'next', 'license', 'djmahe', 'ta...",2.363399
78,brianspiering/awesome-deep-rl,A curated list of awesome Deep Reinforcement L...,[],151,None,https://github.com/brianspiering/awesome-deep-rl,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",bb89d0c8-db30-5379-9b8d-f4b0f5dd3a4f,,Terms(set()),"Terms({'curated', 'awesome', 'list', 'resource...","Terms({'awesome', 'brianspiering', 'deep'})",Awesome Deep Reinforcement Learning [![Awesome...,Awesome Deep Reinforcement Learning ![Awesome]...,"Terms({'agent', 'lunar', 'dopamine', 'lex', 'h...",2.351770
471,zudochkin/awesome-newsletters,A list of amazing Newsletters,"[awesome, awesome-list, email, email-newslette...",4322,None,https://github.com/zudochkin/awesome-newsletters,readme,

In [24]:
awesome_curated_lists = "awesome curated lists"
tokenized_phrase = preprocess_text(awesome_curated_lists)
tokenized_phrase

['awesome', 'curated', 'list']

In [25]:
df['score'] = df['content_tokenized'].array.score(tokenized_phrase)

In [26]:
df.sort_values('score', ascending=False).head(30)

,repo,description,topics,stars,language,url,doc_source,docs,id,topics_processed,topics_tokenized,description_tokenized,repo_tokenized,content,content_normalized,content_tokenized,score
1123,mauhai/awesome-jupyterlab,A curated list of awesome JupyterLab extensio...,"[awesome, collections, jupyterlab, jupyterlab-...",2616,None,https://github.com/mauhai/awesome-jupyterlab,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",5bab555b-9ee4-500e-8f8a-738d2247825b,awesome collections jupyterlab jupyterlab exte...,"Terms({'awesome', 'extension', 'jupyterlab', '...","Terms({'curated', 'awesome', 'list', 'resource...","Terms({'awesome', 'jupyterlab', 'mauhai'})",# Awesome JupyterLab[![Awesome](https://cdn.ra...,Awesome JupyterLab![Awesome](https://github.co...,"Terms({'fasta', 'distribution', 'required', 'b...",3.570782
887,shubhamgrg04/awesome-diagramming,A curated collection of diagramming tools used...,"[awesome-list, diagram, uml]",3225,None,https://github.com/shubhamgrg04/awesome-diagra...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",87f9338c-1ff5-5e64-93c4-24bcfd6ba711,awesome list diagram uml,"Terms({'awesome', 'diagram', 'list', 'uml'})","Terms({'engineering', 'curated', 'leading', 't...","Terms({'awesome', 'shubhamgrg', 'diagramming'})",# Awesome Diagramming [![Awesome](https://cdn....,Awesome Diagramming ![Awesome](https://github....,"Terms({'outdated', 'component', 'hand', 'spark...",3.451823
840,hemansnation/deep-learning-project-ideas,"Curated list of Machine Learning, NLP, Vision,...",[],29,None,https://github.com/hemansnation/deep-learning-...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",1d81f81c-e006-5b24-858c-b705b3148256,,Terms(set()),"Terms({'recommender', 'curated', 'system', 'ma...","Terms({'idea', 'project', 'deep', 'hemansnatio...",<!-- markdownlint-disable MD033 -->\n\n# Aweso...,Awesome Deep Learning Project Ideas\n![Awesome...,"Terms({'million', 'label', 'captioning', 'scho...",2.704698
1373,wuchong/awesome-flink,😎 A curated list of amazingly awesome Flink a...,"[awesome, awesome-flink, awesome-list, flink]",781,None,https://github.com/wuchong/awesome-flink,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",4f86df74-8927-5d0d-8e2c-d050198c5d68,awesome awesome flink awesome list flink,"Terms({'awesome', 'flink', 'list'})","Terms({'curated', 'awesome', 'flink', 'list', ...","Terms({'awesome', 'flink', 'wuchong'})","[<img src=""https://raw.githubusercontent.com/w...",Awesome Flink ![Awesome](https://github.com/si...,"Terms({'unbounded', 'recovery', 'license', 'ta...",2.554654
1019,NirantK/awesome-project-ideas,"Curated list of Machine Learning, NLP, Vision,...","[awesome, awesome-list, classification, datase...",8960,None,https://github.com/NirantK/awesome-project-ideas,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",c64ddcff-35f6-5f3e-b957-23481ef7153c,awesome awesome list classification dataset de...,"Terms({'image', 'awesome', 'multi', 'machine',...","Terms({'recommender', 'curated', 'system', 'ma...","Terms({'awesome', 'idea', 'project', 'nirantk'})",<!-- markdownlint-disable MD033 -->\n\n# Aweso...,Awesome Deep Learning Project Ideas\n![Awesome...,"Terms({'million', 'label', 'captioning', 'blip...",2.509537
1018,costezki/awesome-nlprojects,List of projects related to Natural Language P...,"[awersome-list, machine-learning, natural-lang...",361,None,https://github.com/costezki/awesome-nlprojects,readme,"[{'name': 'readme.md', 'path': 'readme.md', 's...",4060f165-6902-53bc-8e6d-60217fda3ea2,awersome list machine learning natural languag...,"Terms({'language', 'machine', 'list', 'process...","Terms({'language', 'related', 'geek', 'exist',...","Terms({'awesome', 'nlprojects', 'costezki'})",# Awesome NLP Projects\n[![Awesome](https://cd...,Awesome NLP Projects\n![Awesome](https://githu...,"Terms({'input', 'costezki', 'technique', 'appr...",2.161598
713,kengz/awesome-deep-rl,A curated list of awesome Deep Reinforcement L...,"[awesome-list, deep-reinforcement-lear

In [27]:
def multi_match_search(query: str, df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    # Tokenize the query using each column's own tokenizer
    tokenized_queries = {
        col: df[col].array.tokenizer(query)
        for col in columns
    }

    # Score each column for each query term → shape (num_terms x num_docs) per column
    field_scores = {
        col: np.asarray([df[col].array.score(term) for term in tokenized_queries[col]])
        for col in columns
    }

    # Determine the number of terms from the first column
    num_terms = max(len(scores) for scores in field_scores.values())

    # For each term, take the element-wise max score across all columns (dismax)
    best_term_scores_per_doc = []
    for term_idx in range(num_terms):
        term_scores_across_fields = [
            field_scores[col][term_idx]
            for col in columns
            if term_idx < len(field_scores[col])
        ]
        best_score = np.max(term_scores_across_fields, axis=0)
        best_term_scores_per_doc.append(best_score)

    # Sum the best-field scores across all terms for the final document score
    result = df.copy()
    result['score'] = np.sum(best_term_scores_per_doc, axis=0)
    return result.sort_values('score', ascending=False)

In [28]:
results = multi_match_search(
    query="awesome curated lists",
    df=df,
    columns=["repo_tokenized", "topics_tokenized", "description_tokenized", "content_tokenized"],
)

In [32]:
# Show all rows
cols_to_export = ['id','repo', 'description', 'topics', 'language','url', 'doc_source', 'score']
pd.set_option('display.max_rows', 200)
results[cols_to_export].head(20)

,id,repo,description,topics,language,url,doc_source,score
1950,6a6a505a-45ec-5402-b338-aea20a0047fd,jnv/lists,The definitive list of lists (of lists) curate...,"[awesome, awesome-list, curated-lists, lists, ...",None,https://github.com/jnv/lists,readme,8.930840
471,1dbd0153-427b-5c52-9793-faa45ae6877e,zudochkin/awesome-newsletters,A list of amazing Newsletters,"[awesome, awesome-list, email, email-newslette...",None,https://github.com/zudochkin/awesome-newsletters,readme,7.235460
675,62762741-11e6-5e42-9d12-8e8101f7469d,ddotta/awesome-polars,"A curated list of Polars talks, tools, example...","[awesome, awesome-list, collection, curated-li...",None,https://github.com/ddotta/awesome-polars,readme,7.233369
1524,9647593c-9b19-5b13-b2ef-744125cd6f1e,aporia-ai/mlops.toys,"🎲 A curated list of MLOps projects, tools and ...","[awesome, awesome-list, data-science, list, ma...",Vue,https://github.com/aporia-ai/mlops.toys,readme,6.951973
88,7636125b-0534-500b-b0b9-22717eeefb11,awesomelistsio/awesome-reinforcement-learning,"A curated list of awesome frameworks, librarie...","[awesome, awesome-list, awesome-lists, reinfor...",Python,https://github.com/awesomelistsio/awesome-rein...,readme,6.851551
1137,e18afc19-c4d7-5fea-b056-72f20f33f4a3,wsvincent/awesome-django,A curated list of awesome things related to Dj...,"[awesome, awesome-list, django]",Python,https://github.com/wsvincent/awesome-django,readme,6.793023
1881,a4ebee38-74c6-5779-b331-0c15a5d58b9b,pditommaso/awesome-pipeline,A curated list of awesome pipeline toolkits in...,"[awesome-list, workflow]",None,https://github.com/pditommaso/awesome-pipeline,readme,6.739195
285,c4fec4da-7b85-5540-aa31-346a58c8c424,mjhea0/awesome-fastapi,A curated list of awesome things related to Fa...,"[awesome, awesome-list, fastapi, starlette]",None,https://github.com/mjhea0/awesome-fastapi,readme,6.574901
887,87f9338c-1ff5-5e64-93c4-24bcfd6ba711,shubhamgrg04/awesome-diagramming,A curated collection of diagramming tools used...,"[awesome-list, diagram, uml]",None,https://github.com/shubhamgrg04/awesome-diagra...,readme,6.521213
1373,4f86df74-8927-5d0d-8e2c-d050198c5d68,wuchong/awesome-flink,😎 A curated list of amazingly awesome Flink a...,"[awesome, awesome-flink, awesome-list, flink]",None,https://github.com/wuchong/awesome-flink,readme,6.479270


In [30]:
## TODO: tokenize more fields and search with multi-match-search function
results.shape

(2049, 17)

In [35]:
# export to CSV and spreadsheet
results[cols_to_export].sort_values('score', ascending=False).to_csv("../data/processed/github_scores.csv", index=False)

In [38]:
 # Read only specific columns
results_ann = pd.read_csv("../data/processed/github_scores_annotated.csv")
results_ann.head()

,repo,topics,language,id,description,url,doc_source,score,is_a_curated_list,is_not_a_library
0,jnv/lists,"['awesome', 'awesome-list', 'curated-lists', '...",NaN,6a6a505a-45ec-5402-b338-aea20a0047fd,The definitive list of lists (of lists) curate...,https://github.com/jnv/lists,readme,8.930841,1.0,1.0
1,zudochkin/awesome-newsletters,"['awesome', 'awesome-list', 'email', 'email-ne...",NaN,1dbd0153-427b-5c52-9793-faa45ae6877e,A list of amazing Newsletters,https://github.com/zudochkin/awesome-newsletters,readme,7.235460,1.0,1.0
2,ddotta/awesome-polars,"['awesome', 'awesome-list', 'collection', 'cur...",NaN,62762741-11e6-5e42-9d12-8e8101f7469d,"A curated list of Polars talks, tools, example...",https://github.com/ddotta/awesome-polars,readme,7.233369,1.0,1.0
3,aporia-ai/mlops.toys,"['awesome', 'awesome-list', 'data-science', 'l...",Vue,9647593c-9b19-5b13-b2ef-744125cd6f1e,"🎲 A curated list of MLOps projects, tools and ...",https://github.com/aporia-ai/mlops.toys,readme,6.951973,1.0,1.0
4,awesomelistsio/awesome-reinforcement-learning,"['awesome', 'awesome-list', 'awesome-lists', '...",Python,7636125b-0534-500b-b0b9-22717eeefb11,"A curated list of awesome frameworks, librarie...",https://github.com/awesomelistsio/awesome-rein...,readme,6.851551,1.0,1.0


In [39]:
def multi_match_search(
    query: str,
    df: pd.DataFrame,
    columns: list[str],
    boosts: dict[str, float] | None = None
) -> pd.DataFrame:
    # Default all boosts to 1.0 if not provided
    if boosts is None:
        boosts = {}
    boost_values = {col: boosts.get(col, 1.0) for col in columns}

    # Tokenize the query using each column's own tokenizer
    tokenized_queries = {
        col: df[col].array.tokenizer(query)
        for col in columns
    }

    # Score each column for each query term → shape (num_terms x num_docs) per column
    # Apply the boost multiplier to each field's scores here
    field_scores = {
        col: np.asarray([df[col].array.score(term) for term in tokenized_queries[col]]) * boost_values[col]
        for col in columns
    }

    # Determine the number of terms from the longest tokenized query
    num_terms = max(len(scores) for scores in field_scores.values())

    # For each term, take the element-wise max score across all columns (dismax)
    best_term_scores_per_doc = []
    for term_idx in range(num_terms):
        term_scores_across_fields = [
            field_scores[col][term_idx]
            for col in columns
            if term_idx < len(field_scores[col])
        ]
        best_score = np.max(term_scores_across_fields, axis=0)
        best_term_scores_per_doc.append(best_score)

    # Sum the best-field scores across all terms for the final document score
    result = df.copy()
    result['score'] = np.sum(best_term_scores_per_doc, axis=0)
    return result.sort_values('score', ascending=False)

In [41]:
results2 = multi_match_search(
    query="awesome curated lists",
    df=df,
    columns=["repo_tokenized", "topics_tokenized", "description_tokenized", "content_tokenized"],
    boosts={"repo_tokenized": 3.0, "topics_tokenized": 2.0, "description_tokenized": 1.5}
)

In [47]:
# Keep only what you need before joining to avoid column name collisions
comparison = results_ann[['id', 'repo', 'topics', 'description', 'score', 'is_a_curated_list', 'is_not_a_library']].merge(
    results2[['id', 'score']],
    on='id',
    suffixes=('_v1', '_v2')
)

In [54]:
comparison.head(20)

,id,repo,topics,description,score_v1,is_a_curated_list,is_not_a_library,score_v2
0,6a6a505a-45ec-5402-b338-aea20a0047fd,jnv/lists,"['awesome', 'awesome-list', 'curated-lists', '...",The definitive list of lists (of lists) curate...,8.930841,1.0,1.0,20.959242
1,1dbd0153-427b-5c52-9793-faa45ae6877e,zudochkin/awesome-newsletters,"['awesome', 'awesome-list', 'email', 'email-ne...",A list of amazing Newsletters,7.235460,1.0,1.0,11.148891
2,62762741-11e6-5e42-9d12-8e8101f7469d,ddotta/awesome-polars,"['awesome', 'awesome-list', 'collection', 'cur...","A curated list of Polars talks, tools, example...",7.233369,1.0,1.0,14.590599
3,9647593c-9b19-5b13-b2ef-744125cd6f1e,aporia-ai/mlops.toys,"['awesome', 'awesome-list', 'data-science', 'l...","🎲 A curated list of MLOps projects, tools and ...",6.951973,1.0,1.0,12.115523
4,7636125b-0534-500b-b0b9-22717eeefb11,awesomelistsio/awesome-reinforcement-learning,"['awesome', 'awesome-list', 'awesome-lists', '...","A curated list of awesome frameworks, librarie...",6.851551,1.0,1.0,12.145926
5,e18afc19-c4d7-5fea-b056-72f20f33f4a3,wsvincent/awesome-django,"['awesome', 'awesome-list', 'django']",A curated list of awesome things related to Dj...,6.793023,1.0,1.0,12.669821
6,a4ebee38-74c6-5779-b331-0c15a5d58b9b,pditommaso/awesome-pipeline,"['awesome-list', 'workflow']",A curated list of awesome pipeline toolkits in...,6.739195,1.0,1.0,11.817943
7,c4fec4da-7b85-5540-aa31-346a58c8c424,mjhea0/awesome-fastapi,"['awesome', 'awesome-list', 'fastapi', 'starle...",A curated list of awesome things related to Fa...,6.574901,1.0,1.0,12.233577
8,87f9338c-1ff5-5e64-93c4-24bcfd6ba711,shubhamgrg04/awesome-diagramming,"['awesome-list', 'diagram', 'uml']",A curated collection of diagramming tools used...,6.521213,1.0,1.0,11.288593
9,4f86df74-8927-5d0d-8e2c-d050198c5d68,wuchong/awesome-flink,"['awesome', 'awesome-flink', 'awesome-list', '...",😎 A curated list of amazingly awesome Flink a...,6.479270,1.0,1.0,12.135574


In [49]:
comparison.query('score_v1 < 0 and score_v2 > 0')

,id,repo,topics,description,score_v1,is_a_curated_list,is_not_a_library,score_v2


In [56]:
comparison.query('is_a_curated_list == 1').sort_values('score_v2', ascending=True)

,id,repo,topics,description,score_v1,is_a_curated_list,is_not_a_library,score_v2
203,688f34a1-1e63-5699-bfd4-fc39eda17a28,eugeneyan/applied-ml,"['applied-data-science', 'applied-machine-lear...",📚 Papers & tech blogs by companies sharing the...,1.019706,1.0,1.0,1.019706
192,78a29dca-4c58-56a3-b009-ec50a982cb0f,Nyandwi/machine_learning_complete,"['computer-vision', 'data-analysis', 'data-sci...",A comprehensive machine learning repository co...,1.140272,1.0,0.0,1.140272
187,8608f534-a0d4-598b-9eba-fd1c909d0b24,Rishabh-creator601/Books,"['computer-vision', 'cpp', 'deep-learning', 'd...",Books / PDFS / EPUBS for different fields of p...,1.201308,1.0,1.0,1.201308
184,1016bca9-95ad-5e1b-b8c8-a5163f7da5e9,dair-ai/ML-Papers-Explained,[],Explanation to key concepts in ML,1.225630,1.0,1.0,1.225630
152,7144519b-186a-5bd9-bafb-936670346a95,chihming/competitive-recsys,"['collaborative-filtering', 'recommendation-al...",A collection of resources for Recommender Syst...,1.603412,1.0,1.0,1.603412
144,670b5771-2b9d-5720-b1e7-2b59a6705fda,NirDiamant/RAG_Techniques,"['ai', 'langchain', 'llama-index', 'llm', 'llm...",This repository showcases various advanced tec...,1.714251,1.0,1.0,1.714251
136,3f995872-441f-5fde-9635-c13fcc7808c5,CodeCutTech/Data-science,"['articles', 'artificial-intelligence', 'data-...",Collection of useful data science topics along...,1.759038,1.0,1.0,1.759038
134,caf93575-2b5f-5b9e-91f4-e0c348a9431d,ashishps1/learn-ai-engineering,"['agentic-ai', 'agents', 'ai', 'deep-learning'...",Learn AI and LLMs from scratch using free reso...,1.796410,1.0,1.0,1.796410
109,6f9c8926-52e0-51ee-8f70-2088c093cb6b,AkashSingh3031/The-Complete-FAANG-Preparation,"['algorithms', 'aptitude', 'collaborate', 'com...","Dive into this repository, a comprehensive res...",2.107226,1.0,1.0,2.107226
107,61250239-b16b-53bc-af8f-7de5a9e2e3b3,realpython/discover-flask,[],Full Stack Web Development with Flask.,2.134127,1.0,1.0,2.134127


In [53]:
comparison.isna().sum()

id                      0
repo                    0
topics                  0
description            62
score_v1                0
is_a_curated_list    1844
is_not_a_library     1844
score_v2                0
dtype: int64

In [58]:
# load next version of data
with open("../data/cached/github_data2.pkl", "rb") as f:
    starred_repo_data = pickle.load(f)

In [59]:
df2 = pd.DataFrame(starred_repo_data)
df2.head()

,repo,description,topics,stars,language,url,doc_source,docs
0,laurynasjs/personalized-recommendations-at-scale,None,[],2,Jupyter Notebook,https://github.com/laurynasjs/personalized-rec...,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
1,sumitsidana/corise_personalized_recommendation...,None,[],2,Jupyter Notebook,https://github.com/sumitsidana/corise_personal...,None,[]
2,hrjoshi28/corise-recsys,personalized-recommendations-at-scale,[],1,Jupyter Notebook,https://github.com/hrjoshi28/corise-recsys,readme,"[{'name': 'README.md', 'path': 'README.md', 's..."
3,tusharagarwal25/corise-Personalized-Recommenda...,None,[],1,Jupyter Notebook,https://github.com/tusharagarwal25/corise-Pers...,None,[]
4,yudhiesh/corise-Personalized-Recommendations-a...,Course work for the co:rise course Personalize...,[],2,Jupyter Notebook,https://github.com/yudhiesh/corise-Personalize...,None,[]


In [64]:
df2['docs']

0       [{'name': 'README.md', 'path': 'README.md', 's...
1                                                      []
2       [{'name': 'README.md', 'path': 'README.md', 's...
3                                                      []
4                                                      []
                              ...                        
2080    [{'name': 'README.md', 'path': 'README.md', 's...
2081    [{'name': 'README.rst', 'path': 'README.rst', ...
2082    [{'name': 'README.md', 'path': 'README.md', 's...
2083    [{'name': 'README.md', 'path': 'README.md', 's...
2084    [{'name': 'README.md', 'path': 'README.md', 's...
Name: docs, Length: 2085, dtype: object

In [65]:
def extract_url_count(docs_list):
    if not isinstance(docs_list, list) or len(docs_list) == 0:
        return None
    counts = [doc.get('url_count') for doc in docs_list if doc.get('url_count') is not None]
    return sum(counts) if counts else None

In [66]:
df2['url_count'] = df2['docs'].apply(extract_url_count)

In [67]:
df2.head()

,repo,description,topics,stars,language,url,doc_source,docs,url_count
0,laurynasjs/personalized-recommendations-at-scale,None,[],2,Jupyter Notebook,https://github.com/laurynasjs/personalized-rec...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",0.0
1,sumitsidana/corise_personalized_recommendation...,None,[],2,Jupyter Notebook,https://github.com/sumitsidana/corise_personal...,None,[],NaN
2,hrjoshi28/corise-recsys,personalized-recommendations-at-scale,[],1,Jupyter Notebook,https://github.com/hrjoshi28/corise-recsys,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",0.0
3,tusharagarwal25/corise-Personalized-Recommenda...,None,[],1,Jupyter Notebook,https://github.com/tusharagarwal25/corise-Pers...,None,[],NaN
4,yudhiesh/corise-Personalized-Recommendations-a...,Course work for the co:rise course Personalize...,[],2,Jupyter Notebook,https://github.com/yudhiesh/corise-Personalize...,None,[],NaN


In [68]:
df2['id'] = df2['repo'].apply(repo_to_uuid)
df2.head()

,repo,description,topics,stars,language,url,doc_source,docs,url_count,id
0,laurynasjs/personalized-recommendations-at-scale,None,[],2,Jupyter Notebook,https://github.com/laurynasjs/personalized-rec...,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",0.0,5bc8fab8-25ed-580d-b51a-0c1cda593ec9
1,sumitsidana/corise_personalized_recommendation...,None,[],2,Jupyter Notebook,https://github.com/sumitsidana/corise_personal...,None,[],NaN,e2758043-5e0b-50ee-8be2-5296c76fbe36
2,hrjoshi28/corise-recsys,personalized-recommendations-at-scale,[],1,Jupyter Notebook,https://github.com/hrjoshi28/corise-recsys,readme,"[{'name': 'README.md', 'path': 'README.md', 's...",0.0,89169455-9ef8-56ea-951e-f5389db2dc49
3,tusharagarwal25/corise-Personalized-Recommenda...,None,[],1,Jupyter Notebook,https://github.com/tusharagarwal25/corise-Pers...,None,[],NaN,b2bc8a01-a1bb-54a7-99fd-8b398bb4a3d7
4,yudhiesh/corise-Personalized-Recommendations-a...,Course work for the co:rise course Personalize...,[],2,Jupyter Notebook,https://github.com/yudhiesh/corise-Personalize...,None,[],NaN,20cc77ee-c042-52dc-bc36-da25eb309b8a


In [69]:
comparison2 = (
    results_ann[['id', 'repo', 'topics', 'description', 'score', 'is_a_curated_list', 'is_not_a_library']]
    .merge(results2[['id', 'score']],
           on='id',
           suffixes=('_v1', '_v2')
           )
    .merge(df2[['id', 'url_count']], on='id')
)

In [70]:
comparison2.head()

,id,repo,topics,description,score_v1,is_a_curated_list,is_not_a_library,score_v2,url_count
0,6a6a505a-45ec-5402-b338-aea20a0047fd,jnv/lists,"['awesome', 'awesome-list', 'curated-lists', '...",The definitive list of lists (of lists) curate...,8.930841,1.0,1.0,20.959242,1797.0
1,1dbd0153-427b-5c52-9793-faa45ae6877e,zudochkin/awesome-newsletters,"['awesome', 'awesome-list', 'email', 'email-ne...",A list of amazing Newsletters,7.235460,1.0,1.0,11.148891,339.0
2,62762741-11e6-5e42-9d12-8e8101f7469d,ddotta/awesome-polars,"['awesome', 'awesome-list', 'collection', 'cur...","A curated list of Polars talks, tools, example...",7.233369,1.0,1.0,14.590599,649.0
3,9647593c-9b19-5b13-b2ef-744125cd6f1e,aporia-ai/mlops.toys,"['awesome', 'awesome-list', 'data-science', 'l...","🎲 A curated list of MLOps projects, tools and ...",6.951973,1.0,1.0,12.115523,5.0
4,7636125b-0534-500b-b0b9-22717eeefb11,awesomelistsio/awesome-reinforcement-learning,"['awesome', 'awesome-list', 'awesome-lists', '...","A curated list of awesome frameworks, librarie...",6.851551,1.0,1.0,12.145926,52.0


In [71]:
comparison2.query('is_a_curated_list == 1').sort_values('score_v2', ascending=True)

,id,repo,topics,description,score_v1,is_a_curated_list,is_not_a_library,score_v2,url_count
203,688f34a1-1e63-5699-bfd4-fc39eda17a28,eugeneyan/applied-ml,"['applied-data-science', 'applied-machine-lear...",📚 Papers & tech blogs by companies sharing the...,1.019706,1.0,1.0,1.019706,779.0
192,78a29dca-4c58-56a3-b009-ec50a982cb0f,Nyandwi/machine_learning_complete,"['computer-vision', 'data-analysis', 'data-sci...",A comprehensive machine learning repository co...,1.140272,1.0,0.0,1.140272,115.0
187,8608f534-a0d4-598b-9eba-fd1c909d0b24,Rishabh-creator601/Books,"['computer-vision', 'cpp', 'deep-learning', 'd...",Books / PDFS / EPUBS for different fields of p...,1.201308,1.0,1.0,1.201308,188.0
184,1016bca9-95ad-5e1b-b8c8-a5163f7da5e9,dair-ai/ML-Papers-Explained,[],Explanation to key concepts in ML,1.225630,1.0,1.0,1.225630,588.0
152,7144519b-186a-5bd9-bafb-936670346a95,chihming/competitive-recsys,"['collaborative-filtering', 'recommendation-al...",A collection of resources for Recommender Syst...,1.603412,1.0,1.0,1.603412,146.0
144,670b5771-2b9d-5720-b1e7-2b59a6705fda,NirDiamant/RAG_Techniques,"['ai', 'langchain', 'llama-index', 'llm', 'llm...",This repository showcases various advanced tec...,1.714251,1.0,1.0,1.714251,322.0
136,3f995872-441f-5fde-9635-c13fcc7808c5,CodeCutTech/Data-science,"['articles', 'artificial-intelligence', 'data-...",Collection of useful data science topics along...,1.759038,1.0,1.0,1.759038,68.0
134,caf93575-2b5f-5b9e-91f4-e0c348a9431d,ashishps1/learn-ai-engineering,"['agentic-ai', 'agents', 'ai', 'deep-learning'...",Learn AI and LLMs from scratch using free reso...,1.796410,1.0,1.0,1.796410,114.0
109,6f9c8926-52e0-51ee-8f70-2088c093cb6b,AkashSingh3031/The-Complete-FAANG-Preparation,"['algorithms', 'aptitude', 'collaborate', 'com...","Dive into this repository, a comprehensive res...",2.107226,1.0,1.0,2.107226,846.0
107,61250239-b16b-53bc-af8f-7de5a9e2e3b3,realpython/discover-flask,[],Full Stack Web Development with Flask.,2.134127,1.0,1.0,2.134127,75.0


In [72]:
comparison2.query('is_a_curated_list == 0').sort_values('score_v2', ascending=True)

,id,repo,topics,description,score_v1,is_a_curated_list,is_not_a_library,score_v2,url_count
202,e90fb75b-37dd-5f23-bca4-04faec7978f8,gorakhargosh/watchdog,[],Python library and shell utilities to monitor ...,1.027404,0.0,0.0,1.027404,32.0
201,28f5143a-7b06-5cb7-9831-dd5db52a39e6,devinpleuler/analytics-handbook,[],Getting started with soccer analytics,1.032347,0.0,0.0,1.032347,157.0
200,5bc0451b-6390-5127-9b10-a9dd003d0d63,h2oai/h2o-3,"['automl', 'big-data', 'data-science', 'deep-l...","H2O is an Open Source, Distributed, Fast & Sca...",1.064105,0.0,0.0,1.064105,114.0
199,4f55e95e-7ebd-5158-8006-968841169832,a8m/golang-cheat-sheet,"['cheat-sheets', 'cheatsheet', 'go', 'golang']",An overview of Go syntax and features.,1.071749,0.0,1.0,1.071749,6.0
196,463cc271-96c3-56a8-80c5-c1bca2a67cf3,cloneofsimo/lora,"['diffusion', 'dreambooth', 'fine-tuning', 'lo...",Using Low-rank adaptation to quickly fine-tune...,1.081852,0.0,0.0,1.081852,23.0
195,c825ba40-ed88-510a-9afd-ff2bcf71a6cf,pytorch/rl,"['ai', 'control', 'decision-making', 'distribu...","A modular, primitive-first, python-first PyTor...",1.095050,0.0,0.0,1.095050,107.0
194,db58c1c8-3f08-58df-b5f8-70b499d5ee5e,dbrattli/Expression,"['fsharp', 'functional-programming', 'oslash',...",Functional programming for Python,1.100268,0.0,0.0,1.100268,48.0
193,3a5e8997-f932-53b4-b3e6-8ca20d5671c6,jd/tenacity,"['failure', 'hacktoberfest', 'python', 'retry'...",Retrying library for Python,1.106836,0.0,0.0,1.106836,12.0
191,cfc97d5e-7a05-55a6-9d13-9c9f79390312,MilaNLProc/contextualized-topic-models,"['bert', 'embeddings', 'multilingual-models', ...",A python package to run contextualized topic m...,1.144430,0.0,0.0,1.144430,61.0
188,7199ac5d-6d13-5de2-82c6-6671f3805f7e,omkarcloud/botasaurus,"['anti-bot', 'anti-detect', 'anti-detect-brows...",The All in One Framework to Build Undefeatable...,1.159648,0.0,0.0,1.159648,231.0


- having a URL count is another indicator of a curated list, not necessarily a legitimate repository that could be messing up results of user queries
- we utilize a BM25 scoring filter (IE `score_v2` boosting logic) stored on the index will be important for ignoring curated lists at retrieval time)
- 2nd filter on curated lists can be based on LLM as a judge (use an LLM to determine given all the data we have on the repo whether it is a curated list or not; can use a small LLM for this task so that it is fast)